In [2]:
from pyspark.sql import SparkSession

spark = (SparkSession
            .builder
            .master("local")
            .appName("kafka-streaming")
            .config("spark.jars.packages","org.apache.spark:spark-sql-kafka-0-10_2.11:2.4.5")
            .getOrCreate())

ModuleNotFoundError: No module named 'pyspark'

In [ ]:
df = (spark
      .readStream
      .format("kafka")
      .option("kafka.bootstrap.servers", "kafka:9092")
      .option("subscribe", "ingestion-topic")
      .load())

In [ ]:
df.createOrReplaceTempView("message")

In [ ]:
res = spark.sql("SELECT * FROM message")
res.writeStream.format("console").outputMode("append").start()

In [ ]:
ds = (df
      .writeStream
      .format("kafka")
      .option("kafka.bootstrap.servers", "kafka:9092")
      .option("topic", "spark-output")
      .option("checkpointLocation", "/tmp")
      .start()
      .awaitTermination()
    )